# Notebook 02 — Data Cleaning & Quality

**Tugas 1 · Analisis Big Data · Semester Ganjil 2026**

Notebook ini dipakai pada **Milestone 2**:

1. Profiling dengan `DuckDB SUMMARIZE`.
2. Menangani missing value, outlier, dan duplikat dengan Polars.
3. Standardisasi teks dan validasi dasar.
4. Menyimpan dataset bersih ke `data/processed/`.


## 1. Muat Data

> Sesuaikan nama kolom pada contoh di bawah dengan datasetmu.


In [ ]:
import duckdb
import polars as pl

from src.data_loading import hitung_missing, resolve_data_path

lf = pl.scan_parquet(resolve_data_path("raw/dataset.parquet"))
df = lf.collect()
print(f"{df.height:,} baris x {df.width} kolom")


## 2. Profiling (DuckDB SUMMARIZE)


In [ ]:
duckdb.sql("SUMMARIZE SELECT * FROM df").df()


In [ ]:
hitung_missing(df)


## 3. Missing Value

Strategi bergantung pada persentase missing & jenis kolom:
median/mean untuk numerik, modus untuk kategorik, atau flag lalu buang.


In [ ]:
# TODO: ganti nama kolom sesuai dataset
df_clean = df.with_columns(
    [
        pl.col("kolom_numerik").fill_null(pl.col("kolom_numerik").median()),
        pl.col("kolom_kategorik").fill_null("tidak_diketahui"),
    ]
)
hitung_missing(df_clean)


## 4. Outlier (Metode IQR + Domain Knowledge)


In [ ]:
# TODO: ganti "harga" dengan kolom numerik yang dianalisis
kolom = "harga"
q1 = df_clean[kolom].quantile(0.25)
q3 = df_clean[kolom].quantile(0.75)
iqr = q3 - q1
batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Batas IQR: {batas_bawah:,.0f} s/d {batas_atas:,.0f}")


In [ ]:
# Contoh: clamp (windsorization) + flag outlier
df_clean = df_clean.with_columns(
    [
        pl.col(kolom).clip(batas_bawah, batas_atas).alias(f"{kolom}_clean"),
        ((pl.col(kolom) < batas_bawah) | (pl.col(kolom) > batas_atas)).alias("is_outlier"),
    ]
)
df_clean["is_outlier"].sum()


## 5. Standardisasi Teks


In [ ]:
# TODO: sesuaikan daftar kolom teks
df_clean = df_clean.with_columns(
    [
        pl.col("kota").str.strip_chars().str.to_titlecase(),
    ]
)
df_clean


## 6. Duplikat


In [ ]:
# TODO: ganti subset dengan kolom kunci (primary key) datasetmu
df_clean = df_clean.unique(subset=["id"], keep="first", maintain_order=True)
print(f"Baris setelah deduplikasi: {df_clean.height:,}")


## 7. Validasi Dasar

Aturan validasi sederhana sebagai jaring pengaman sebelum data dipakai.


In [ ]:
con = duckdb.connect()
con.register("bersih", df_clean)

con.execute(
    """
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS id_null,
        SUM(CASE WHEN harga_clean <= 0 THEN 1 ELSE 0 END) AS harga_invalid
    FROM bersih
    """
).df()


## 8. Simpan Data Bersih


In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)
df_clean.write_parquet("data/processed/dataset_bersih.parquet")
print("Tersimpan: data/processed/dataset_bersih.parquet")


## Ringkasan

> Catat jumlah baris sebelum/sesudah dan masalah yang berhasil diperbaiki.

---

**AI Disclosure:** catat penggunaan AI asisten untuk notebook ini di `README.md`.
